# Initializing the environment

In [1]:
from gymnasium import make
from samples.llm_interface import SoloGPT4Interfacer, MultiGPT4Interfacer
import os
import random
from natural20.gym.tools import compute_available_moves
from experiment_object import Experiment
os.environ["OPENAI_API_KEY"] = ""

n_player1 = 2
n_player2 = 2
# Initialize the environment
# env = make("dndenv-v0", root_path="templates", render_mode="ansi")
all_classes = ['halfling_rogue.yml', 'high_elf_fighter.yml']
all_players = ["Alysha", "Bernard", "Cedric", "Didier", "Eric", "Francois", "Gertrude", "Heloise", "Isabelle"]
players = random.sample(all_players, n_player1 + n_player2)
a_player = [(random.choice(all_classes), player) for player in players[:n_player1]]
e_player = [(random.choice(all_classes), player) for player in players[n_player1:]]

conversational_groups = {"a":True, "b":False}
env = make(
    "dndenv-v0",
    render_mode="ansi",
    map_file="maps/game_map.yml",
    show_logs=True,
    profiles=a_player,
    enemies=e_player,
    control_groups=["a","b"]
    )

envi = env.env.env
observation, info = env.reset(seed = random.randint(0,1000), options={"initial_poses":[[[3,3], [5, 3]], [[3, 5], [5,5]]]})



agents = {}
groups = env.env.env.battle.groups  
for character in env.env.env.battle.combat_order:
    gr = None
    for group_name, group in env.env.env.battle.groups.items():
        if character in group:
            gr = group_name
    if gr == None : print(f"Warning ! : Character {character.name} has no friends !!! (aka no groups attributed)")
    agent_type = MultiGPT4Interfacer if conversational_groups[gr] else SoloGPT4Interfacer
    agents[character.name] = (agent_type(debug=True, explain=True, name=character.name), gr, character)

agents


{'seed': 988, 'options': {'initial_poses': [[[3, 3], [5, 3]], [[3, 5], [5, 5]]]}}
Alysha rolled initiative d20(10) + 5 value 15.2
Bernard rolled initiative d20(20) + 5 value 25.2
Heloise rolled initiative d20(4) + 5 value 9.2
Isabelle rolled initiative d20(19) + 5 value 24.2
Alysha rolled initiative d20(4) + 5 value 9.2
Bernard rolled initiative d20(8) + 5 value 13.2
Heloise rolled initiative d20(3) + 5 value 8.2
Isabelle rolled initiative d20(10) + 5 value 15.2
Combat begins with 4 players.
Players: <p>Alysha (fighter-2) Team a</p>
<p>Bernard (fighter-2) Team a</p>
<p>Heloise (rogue-2) Team b</p>
<p>Isabelle (rogue-2) Team b</p>
======== Isabelle starts their turn. ========
======== Isabelle starts their turn. ========


/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(
/home/thomas/.conda/envs/dandllm/lib/python3.13/site-packages/gymnasium/spaces/box.py:423: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


{'Isabelle': (<samples.llm_interface.SoloGPT4Interfacer at 0x7efe0ce70440>,
  'b',
  Isabelle),
 'Bernard': (<samples.llm_interface.MultiGPT4Interfacer at 0x7efe0cc41be0>,
  'a',
  Bernard),
 'Alysha': (<samples.llm_interface.MultiGPT4Interfacer at 0x7efe0cc4efd0>,
  'a',
  Alysha),
 'Heloise': (<samples.llm_interface.SoloGPT4Interfacer at 0x7efe0cc57890>,
  'b',
  Heloise)}

In [2]:
expe.dnd_environment.players

NameError: name 'expe' is not defined

In [3]:
expe = Experiment(env, env.env.env, agents, conversational_groups=conversational_groups, debug=False)
expe.run_till_end(max_step=20)

prompt: -------------------------------
We are playing a game of Dungeons and Dragons 5th Edition. It is current your turn and you play 
as a hero character denoted by I (a level 2 rogue).Your health is at [100.]% specifically 16/16 
Your current conditions are:

You have as enemies :
 - Alysha denoted by A (a level 2 fighter).
    Their health is currently at 100.0%.
    Their current conditions are: 
 - Bernard denoted by B (a level 2 fighter).
    Their health is currently at 100.0%.
    Their current conditions are: 
You must defeat all of them in order to win.

You are helped in that regard by your allies :
 - Heloise denoted by H (a level 2 rogue).
    Their health is currently at 100.0%.
    Their current conditions are: You have the following available actions and movement available:

Available movement: [25]ft
Available actions: 1
Bonus actions: 1
Reactions: 1



Here is a rough sketch of the map that considers line of sight to the enemy.
Here is the map:
____________
_.......

/home/thomas/Documents/ENS/IAS/S2/Neural Networks/DandLLM/natural_20.py/samples/llm_interface.py:905: SyntaxWarning: invalid escape sequence '\d'
  regex = "\d"


APIConnectionError: Connection error.

In [4]:
from natural20.gym.llm_helpers.metrics import combat_metrics, combat_score


expe.backlog
metrics = combat_metrics(expe.dnd_environment, agents)
print(metrics)
score = combat_score(metrics)
print(score)

{'win': False, 'turns_taken': 20, 'survivors': {'Eric': (24, 24), 'Isabelle': (3, 16)}}
-8.125


In [ ]:
metrics

{'win': False,
 'turns_taken': 115,
 'survivors': {'Francois': (0, 24),
  'Alysha': (0, 16),
  'Eric': (5, 24),
  'Bernard': (24, 24)}}

In [ ]:
expe.conversations

[[('Alysha',
   "Francois, I'm badly hurt and staying hidden for now. Focus on keeping Eric and Bernard busy—I'll look for a moment to strike safely. Try to draw their attention if you can!"),
  ('Francois',
   "Understood, Alysha! I'll keep the pressure on Eric and try to draw both enemies' attention. Stay safe—strike whenever you see an opening. If they split up, I'll call it out so we can take advantage. Let's keep each other posted."),
  (None, 'Conversation closed')],
 [('Francois',
   "Alysha, Eric is badly hurt but is Dodging right now, making him harder to hit. Stay safe and ready—we'll finish him as soon as an opportunity opens up. Once Eric is down, we both shift our focus to Bernard. Let me know if you need support or see a better move."),
  ('Alysha',
   'Thanks, Francois. I’m keeping my head down for now—no openings yet, but I’m watching for any slip in Eric’s defense. If he drops his guard, I’ll move in for a Sneak Attack. If things get dicey or you need a distraction, gi